# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaminari19/FlyRank-Starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#Configurations

# ============================================================
# CAPSTONE
# A Leakage-Safe Content Opportunity Scoring Model
# for Prioritizing SEO Content Reviews
# ============================================================

import os
import sys
import json
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Main modeling windows
FEATURE_WINDOW_DAYS = 90
MOMENTUM_WINDOW_DAYS = 30
FUTURE_WINDOW_DAYS = 30

# Editorial capacity
TOP_K_VALUES = [20, 50, 100]

print("Capstone configuration loaded.")
print(f"Feature window: {FEATURE_WINDOW_DAYS} days")
print(f"Future outcome window: {FUTURE_WINDOW_DAYS} days")
print(f"Random state: {RANDOM_STATE}")

Capstone configuration loaded.
Feature window: 90 days
Future outcome window: 30 days
Random state: 42


## 1. Question

*The research question and the decision it supports.*

In [2]:
research_question = (
    "Can observable search, content, and engagement signals from "
    "a page's previous 90 days be used to prioritize pages that "
    "are likely to experience meaningful search-visibility decline "
    "in the following 30 days?"
)

decision_supported = (
    "Which content pages should an editor review first when "
    "editorial capacity is limited?"
)

unit_of_analysis = (
    "One content page for one decision date, identified by "
    "(client_hash_id, content_hash_id, decision_date)."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

print("\nUnit of analysis:")
print(unit_of_analysis)

Research question:
Can observable search, content, and engagement signals from a page's previous 90 days be used to prioritize pages that are likely to experience meaningful search-visibility decline in the following 30 days?

Decision supported:
Which content pages should an editor review first when editorial capacity is limited?

Unit of analysis:
One content page for one decision date, identified by (client_hash_id, content_hash_id, decision_date).


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [3]:
# ============================================================
# DATA CONNECTION
# ============================================================

from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your Hugging Face token through "
        "Colab Secrets or the HF_TOKEN environment variable."
    )

login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"

con = duckdb.connect()

con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

con.sql(
    f"""
    CREATE OR REPLACE SECRET hf_token
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
    """
)

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [4]:
# ============================================================
# WAREHOUSE FILE INVENTORY
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

files = sorted(
    api.list_repo_files(
        REPO_ID,
        repo_type="dataset"
    )
)

for f in files:
    print(f)


# ============================================================
# IDENTIFY AVAILABLE DAILY PERFORMANCE MONTHS
# ============================================================

performance_files = [
    f for f in files
    if f.startswith("fact_content_daily_performance/")
]

months = sorted(
    set(
        f.split("month=")[1].split("/")[0]
        for f in performance_files
        if "month=" in f
    )
)

print("Available monthly partitions:")
for month in months:
    print(" -", month)

print("\nNumber of monthly partitions:", len(months))


# ============================================================
# MONTHLY PARTITION SIZE AUDIT
# ============================================================

partition_counts = []

for month in months:

    path = (
        f"hf://datasets/{REPO_ID}/"
        f"fact_content_daily_performance/"
        f"month={month}/*.parquet"
    )

    n = con.sql(
        f"""
        SELECT COUNT(*) AS n
        FROM read_parquet('{path}')
        """
    ).fetchone()[0]

    partition_counts.append({
        "month": month,
        "rows": n
    })

partition_counts_df = pd.DataFrame(partition_counts)

display(partition_counts_df)


# ============================================================
# DATE COVERAGE AUDIT
# ============================================================

coverage = []

for month in months:

    path = (
        f"hf://datasets/{REPO_ID}/"
        f"fact_content_daily_performance/"
        f"month={month}/*.parquet"
    )

    result = con.sql(
        f"""
        SELECT
            MIN(report_date) AS first_date,
            MAX(report_date) AS last_date,
            COUNT(DISTINCT client_hash_id) AS clients,
            COUNT(DISTINCT content_hash_id) AS contents
        FROM read_parquet('{path}')
        """
    ).df()

    result.insert(0, "month", month)
    coverage.append(result)

coverage_df = pd.concat(coverage, ignore_index=True)

display(coverage_df)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,first_date,last_date,clients,contents
0,2025-01,2025-01-27,2025-01-31,2,476
1,2025-02,2025-02-01,2025-02-28,3,5903
2,2025-03,2025-03-01,2025-03-31,4,10374
3,2025-04,2025-04-01,2025-04-30,4,13046
4,2025-05,2025-05-01,2025-05-31,4,14887
5,2025-06,2025-06-01,2025-06-30,9,16399
6,2025-07,2025-07-01,2025-07-31,16,27945
7,2025-08,2025-08-01,2025-08-31,15,37204
8,2025-09,2025-09-01,2025-09-30,23,53127
9,2025-10,2025-10-01,2025-10-31,31,110339


In [5]:
# ============================================================
# DATA AVAILABILITY AUDIT
# ============================================================

availability_results = []

for month in months:

    path = (
        f"hf://datasets/{REPO_ID}/"
        f"fact_content_daily_performance/"
        f"month={month}/*.parquet"
    )

    result = con.sql(
        f"""
        SELECT
            COUNT(*) AS total_rows,

            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
                AND client_has_gsc IS TRUE
            ) AS gsc_available_rows,

            COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
                AND client_has_ga4 IS TRUE
            ) AS ga4_available_rows,

            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
                AND client_has_gsc IS TRUE
                AND ga4_data_available IS TRUE
                AND client_has_ga4 IS TRUE
            ) AS both_available_rows

        FROM read_parquet('{path}')
        """
    ).df()

    result.insert(0, "month", month)
    availability_results.append(result)

availability_df = pd.concat(
    availability_results,
    ignore_index=True
)

display(availability_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,2025-01,1297,1297,0,0
1,2025-02,75985,75985,0,0
2,2025-03,167859,167859,0,0
3,2025-04,285114,285114,0,0
4,2025-05,349923,349923,0,0
5,2025-06,329201,329201,0,0
6,2025-07,469794,469794,0,0
7,2025-08,704962,704962,0,0
8,2025-09,845813,845813,0,0
9,2025-10,2165471,1187654,9545,6666


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
